# Notebook 04: CNN trên ký tự, bỏ dòng trùng, bỏ tiền tố `www.`

Một mô hình đọc thẳng chuỗi URL có tránh được những gì 21 đặc trưng mắc phải không; và các kết luận có đứng vững khi bộ dữ liệu bỏ các host bị ghi hai lần, hay
khi bỏ tiền tố `www.`. Đây là notebook chậm nhất (huấn luyện CNN). Cần chạy notebook 01 và 02 trước. Ô cuối cộng các phép kiểm của cả bốn notebook.

| Notebook | Nội dung | Đọc kết quả của |
|---|---|---|
| `01_benchmark_and_protocol.ipynb` | benchmark bảy họ, chia theo thời gian và các đối chứng, dòng trùng, tầng nhãn | |
| `02_transfer.ipynb` | ma trận chuyển giao giữa bốn corpus, huấn luyện gộp, khoảng tin cậy, SHAP | 01 |
| `03_robustness.ipynb` | tối ưu siêu tham số, stacking, dự báo độ trôi, nhãn nhiễu | 01 |
| `04_cnn_and_dedup.ipynb` | CNN trên ký tự, bỏ dòng trùng, bỏ tiền tố `www.`, tổng hợp cả bốn notebook | 01, 02 |

## Bước 0: Thiết lập

`setup()` (trong `p2lib/setup.py`) đọc các biến môi trường ở trên, tải bộ PhishVN từ Mendeley Data và ba corpus ngoài từ Zenodo, kiểm mã SHA-256 của từng file,
dựng cây thư mục `url_work/data/processed/` mà mã của bài đọc và ghi, rồi chuyển thư mục làm việc vào đó. Các file đã tải ở notebook trước không tải lại. Mọi kết
quả nằm trong `url_work/data/processed/p2/`, và các notebook sau đọc kết quả của notebook trước từ đó.

In [ ]:
import pandas as pd
from threadpoolctl import threadpool_limits

import p2lib
from p2lib import cnn, dedup, derived
from p2lib.checks import RES, MET_TOL, CURVE_TOL, THR_TOL, ROUND4_TOL, CNN_TOL, check, check_table, check_matrix, check_json, report

env = p2lib.setup.setup()
p2lib.setup.require(f"{RES}/p2_temporal_strict.csv", notebook="01")
p2lib.setup.require(f"{RES}/cross_dataset_F1.csv", f"{RES}/cross_dataset_ROC-AUC.csv", f"{RES}/cross_dataset_F1_CatBoost.csv", notebook="02")

## Bước 1: CNN trên ký tự

Một mạng CNN đọc thẳng chuỗi URL thay cho 21 đặc trưng: mỗi ký tự thành một vector 32 chiều, bốn bộ lọc tích chập độ rộng 3 tới 6 (128 bộ mỗi độ rộng), lấy cực
đại theo chiều dài, rồi một lớp tuyến tính (`make_net`). Bảng chữ cái chỉ lấy từ tập train; 10% tập train làm validation để dừng sớm (tối đa 12 epoch, dừng khi
hai epoch liên tiếp không tốt hơn). Seed cố định cả khởi tạo lẫn thứ tự lô, và mạng chạy trên CPU ở chế độ tất định.

Ba lượt trên hai giao thức của notebook 01 (Bước 3): nguyên URL; bỏ phần scheme (`http://`), vì tỷ lệ https khác nhau giữa hai lớp là dấu vết của cách thu thập; và bỏ scheme với
benign chia theo tên miền. Rồi phép thử nguồn benign (`with_source`): huấn luyện với benign chỉ từ một họ nguồn (Tranco, hoặc danh sách Tín Nhiệm Mạng), kiểm tra
với benign từ họ kia, cho CNN, CatBoost và hồi quy logistic, 3 seed và 8 epoch.

Lượt nguyên URL chạy trên CPU như bài, nên được so theo dung sai thông thường. Hai lượt bỏ scheme được bài huấn luyện trên GPU Apple (MPS); trên CPU, phép toán số
thực khác thứ tự nên trọng số học được lệch dần và chỉ số lệch ở chữ số thứ ba, vì vậy được so với dung sai `CNN_TOL` và notebook in mức lệch thật.

In [ ]:
cnn.run_charcnn()
check_table("1 character CNN", "p2_charcnn.csv", MET_TOL)
cnn.run_charcnn(strip_scheme=True, out=f"{RES}/p2_charcnn_noscheme.csv", curves="")
check_table("1 character CNN", "p2_charcnn_noscheme.csv", CNN_TOL)
cnn.run_charcnn(strip_scheme=True, benign_grouped=True, out=f"{RES}/p2_charcnn_noscheme_bgroup.csv", curves="")
check_table("1 character CNN", "p2_charcnn_noscheme_bgroup.csv", CNN_TOL)
cnn.run_source_probe()
check_table("1 character CNN", "p2_source_probe.csv", MET_TOL)
report("1 character CNN")

## Bước 2: CNN trên ký tự trong ma trận chuyển giao

Ma trận của notebook 02 (Bước 1) với mạng CNN thay cho các họ trên đặc trưng: ô đường chéo dùng đúng cách chia trong corpus của notebook 02; ô ngoài đường chéo huấn luyện một mạng
trên toàn bộ corpus nguồn rồi chấm cả ba corpus đích. Bài huấn luyện 40 mạng này trên MPS, nên hai ma trận F1 và ROC-AUC được so với dung sai `CNN_TOL`.

In [ ]:
cnn.run_charcnn_xdata()
check_matrix("2 CNN transfer", "cross_dataset_F1_CharCNN.csv", CNN_TOL)
check_matrix("2 CNN transfer", "cross_dataset_ROC-AUC_CharCNN.csv", CNN_TOL)
report("2 CNN transfer")

## Bước 3: Bỏ dòng trùng và chạy lại

Bộ PhishVN đã công bố ghi 16.855 host hai lần: gần như mọi dòng lừa đảo không có ngày từ nguồn cộng đồng là bản có `www.` của một dòng có ngày cùng host. Bước này
giữ mỗi host một dòng (`dedup`: host đã bỏ scheme và `www.`; ưu tiên dòng có ngày, rồi dòng không có `www.`), còn 36.261 dòng, rồi chạy lại trên corpus đó những
thí nghiệm mà dòng trùng chạm tới: benchmark và chia theo thời gian của notebook 01, đối chứng có guard, ba ma trận chính, rò rỉ trùng lặp, SHAP, rồi các ma trận can thiệp, huấn luyện gộp và CNN chuyển
giao.

Cách làm: dựng một cây thư mục thứ hai `url_work_dedup/` giống hệt cây chính nhưng `dataset_url.csv` và `vn_compphish.csv` là bản đã bỏ trùng, chuyển vào đó và gọi
lại đúng các hàm của các bước trước, với BLAS một luồng như bài. Kết quả được chép sang `data/processed/p2/dedup_audit/` của cây chính và so với bản kỳ vọng cùng
tên.

Toàn bộ việc này nằm trong `dedup.run_dedup_audit`; hàm trả danh sách các bảng nó đã chép để ô dưới so.

In [ ]:
import filecmp
from pathlib import Path

DEDUP_OUT = Path(RES) / "dedup_audit"
names = dedup.run_dedup_audit(env.work, env.home / "url_work_dedup")
for f in names:
    name = f"dedup_audit/{f}.csv"
    if f.startswith("cross_dataset_"):
        check_matrix("3 deduplicated corpus", name, CNN_TOL if f.endswith("_CharCNN") else 0.0, path=DEDUP_OUT / f"{f}.csv")
    elif f != "p2_charcnn_xdata":
        check_table("3 deduplicated corpus", name, MET_TOL, path=DEDUP_OUT / f"{f}.csv")
for f in ("dataset_url_dedup.csv", "vn_compphish_dedup.csv"):
    check("3 deduplicated corpus", f"dedup_audit/{f} byte-identical",
          filecmp.cmp(DEDUP_OUT / f, env.expected / "dedup_audit" / f, shallow=False))
report("3 deduplicated corpus")

## Bước 4: Bỏ tiền tố `www.`

Tiền tố `www.` mang nghĩa ngược nhau ở các corpus: ở PhishVN và ISCX-URL2016 nó gần như chỉ có ở dòng lừa đảo, ở PhiUSIIL nó có ở mọi dòng benign, ở PhishStorm ở
hai phần ba dòng benign. `subdom_cnt`, `dot_cnt` và các đặc trưng độ dài đều đếm nó, nên một mô hình có thể học "có `www.` là lừa đảo" trên corpus này rồi bị chấm
trên corpus mà nó có nghĩa ngược lại. Bước này bỏ tiền tố khỏi mọi host của cả bốn corpus (PhishVN dùng bản mỗi host một dòng của Bước 3), tính lại đặc trưng
(`extract`), ghi tỷ lệ dòng có tiền tố theo corpus và lớp, rồi chạy lại các ma trận chính.

In [ ]:
dedup.run_nowww()
NOWWW = dedup.NOWWW_OUT
check_table("4 no www prefix", "nowww/www_share.csv", MET_TOL, path=f"{NOWWW}/www_share.csv")
check_table("4 no www prefix", "nowww/phishvn_nowww.csv", MET_TOL, path=f"{NOWWW}/phishvn_nowww.csv")
for f in ("cross_dataset_F1.csv", "cross_dataset_ROC-AUC.csv", "cross_dataset_MCC.csv"):
    check_matrix("4 no www prefix", f"nowww/{f}", 0.0, path=f"{NOWWW}/{f}")
report("4 no www prefix")

## Bước 5: Các con số tính từ kết quả

Phần lớn con số của bài là trung bình và độ lệch đọc thẳng từ các bảng đã so ở trên. Một số khác được tính thêm từ các bảng đó hoặc từ chính corpus: kiểm định
có cặp, bootstrap, tương quan, và vài phép đếm lại. `derived.cnn_dedup_numbers` (trong `p2lib/derived.py`) tính lại những con số thuộc các bảng của notebook này và so
từng nhóm với `results/p2_generated.json`, file bài ghi các con số đó ra. Bốn notebook cùng phủ hết các nhóm của file đó; notebook 04 kiểm lại điều này.

In [ ]:
R = derived.cnn_dedup_numbers()
for key in R:
    check_json("5 derived numbers", R[key], "results/p2_generated.json", key=key, tol=MET_TOL)
report("5 derived numbers")

## Bước 6: Tổng hợp cả bốn notebook

Ô dưới lưu các phép kiểm của notebook này, rồi đọc các phép kiểm mà ba notebook trước đã lưu và cộng lại theo notebook. Nó cũng kiểm rằng mọi nhóm con số
trong `results/p2_generated.json` đã được một notebook nào đó so, để không nhóm nào bị bỏ sót khi tách notebook.

In [ ]:
import json
from pathlib import Path

p2lib.checks.save("04")
alls = pd.concat([pd.read_csv(p) for p in sorted(Path(RES, "notebook_checks").glob("*.csv"))], ignore_index=True)
print(alls.groupby("notebook")["match"].agg(checks="size", matched="sum").to_string())
saved_keys = set(json.loads((env.expected / "results/p2_generated.json").read_text()))
checked = set(alls.check.str.extract(r"^results/p2_generated\.json\[(.+)\]$")[0].dropna())
print("groups of p2_generated.json that no notebook compared:", sorted(saved_keys - checked) or "none")
print(f"total: {int(alls['match'].sum())}/{len(alls)} checks match across {alls.notebook.nunique()} notebooks")